# Notebook 1: Journal List & Rankings

**Replication of:** "No Interest: The Marginalization of Women in Academic Finance" (Fine, Yadav & Murawski, 2025)

This notebook builds the master list of finance journals with ABDC quality rankings and maps them to OpenAlex source IDs.

**Steps:**
1. Define the list of finance journals based on ABDC FoR 3502 (Banking, Finance, Investment)
2. Match journals to OpenAlex source IDs
3. Output: `data/processed/journal_list.csv`

In [1]:
import pandas as pd
import pyalex
from pyalex import Sources
from difflib import SequenceMatcher
from tqdm.notebook import tqdm
import time
import os

# Configure OpenAlex polite pool — replace with your email
pyalex.config.email = "your_email@example.com"

print("Libraries loaded.")

Libraries loaded.


## 1. Define Finance Journals

The paper analyzed 171 finance journals listed in the ABDC (Australian Business Deans Council) journal quality list under Field of Research (FoR) code 3502 — Banking, Finance and Investment. Journals are ranked A*, A, B, or C.

Below we define a curated list of major finance journals with their ABDC rankings. This list covers the core journals in the field. If you have the ABDC Excel file, you can load it directly instead (see commented code below).

In [2]:
# ============================================================
# Option A: Load from ABDC Excel file (if you have it)
# Download from: https://abdc.edu.au/abdc-journal-quality-list/
# Save to data/external/abdc_journal_list.xlsx
# ============================================================
# abdc_path = "data/external/abdc_journal_list.xlsx"
# if os.path.exists(abdc_path):
#     abdc_df = pd.read_excel(abdc_path)
#     # Filter to finance FoR codes
#     finance_fors = ["3502"]  # Banking, Finance and Investment
#     finance_journals = abdc_df[
#         abdc_df["FoR"].astype(str).str.contains("|".join(finance_fors))
#     ][["Journal Title", "ABDC Rating"]].copy()
#     finance_journals.columns = ["journal_name", "abdc_rank"]
#     print(f"Found {len(finance_journals)} finance journals from ABDC list")

# ============================================================
# Option B: Curated list of finance journals with ABDC rankings
# This covers the major journals used in the paper
# ============================================================

finance_journals_data = [
    # A* journals
    ("Journal of Finance", "A*"),
    ("Journal of Financial Economics", "A*"),
    ("Review of Financial Studies", "A*"),
    ("Journal of Monetary Economics", "A*"),
    ("Journal of Financial and Quantitative Analysis", "A*"),
    ("Review of Finance", "A*"),
    ("Journal of Money, Credit and Banking", "A*"),
    ("Mathematical Finance", "A*"),
    ("Finance and Stochastics", "A*"),
    ("Journal of Financial Intermediation", "A*"),
    
    # A journals
    ("Journal of Banking and Finance", "A"),
    ("Journal of Corporate Finance", "A"),
    ("Journal of Empirical Finance", "A"),
    ("Journal of International Money and Finance", "A"),
    ("Financial Analysts Journal", "A"),
    ("Journal of Futures Markets", "A"),
    ("Journal of Risk and Insurance", "A"),
    ("Journal of Financial Markets", "A"),
    ("Journal of Financial Research", "A"),
    ("Financial Management", "A"),
    ("Journal of Financial Services Research", "A"),
    ("Journal of Derivatives", "A"),
    ("Review of Derivatives Research", "A"),
    ("Journal of Portfolio Management", "A"),
    ("European Financial Management", "A"),
    ("Journal of International Financial Markets, Institutions and Money", "A"),
    ("Financial Review", "A"),
    ("Journal of Fixed Income", "A"),
    ("Journal of Financial Stability", "A"),
    ("International Review of Financial Analysis", "A"),
    ("Journal of Asset Management", "A"),
    ("Quantitative Finance", "A"),
    ("Journal of Banking Regulation", "A"),
    ("Journal of Pension Economics and Finance", "A"),
    ("Pacific-Basin Finance Journal", "A"),
    ("Annals of Finance", "A"),
    ("Journal of Financial Regulation and Compliance", "A"),
    ("Journal of Behavioral Finance", "A"),
    ("North American Journal of Economics and Finance", "A"),
    ("International Review of Economics and Finance", "A"),
    
    # B journals
    ("Finance Research Letters", "B"),
    ("Journal of Multinational Financial Management", "B"),
    ("Emerging Markets Review", "B"),
    ("International Journal of Finance and Economics", "B"),
    ("Global Finance Journal", "B"),
    ("Journal of International Financial Management and Accounting", "B"),
    ("Applied Financial Economics", "B"),
    ("Managerial Finance", "B"),
    ("Quarterly Review of Economics and Finance", "B"),
    ("Journal of Economics and Finance", "B"),
    ("Review of Quantitative Finance and Accounting", "B"),
    ("International Finance", "B"),
    ("Journal of Financial Crime", "B"),
    ("Journal of Financial Planning", "B"),
    ("Research in International Business and Finance", "B"),
    ("Review of Pacific Basin Financial Markets and Policies", "B"),
    ("Journal of Financial Regulation", "B"),
    ("Journal of Financial Management, Markets and Institutions", "B"),
    ("Journal of Risk", "B"),
    ("Journal of Financial Econometrics", "B"),
    ("Asian Journal of Finance and Accounting", "B"),
    ("Journal of Investing", "B"),
    ("Borsa Istanbul Review", "B"),
    ("Journal of Risk and Financial Management", "B"),
    ("International Journal of Banking, Accounting and Finance", "B"),
    ("Journal of Financial Counseling and Planning", "B"),
    ("Financial Markets and Portfolio Management", "B"),
    ("Journal of Structured Finance", "B"),
    ("Journal of Trading", "B"),
    ("Emerging Markets Finance and Trade", "B"),
    
    # C journals
    ("International Journal of Monetary Economics and Finance", "C"),
    ("Journal of Financial Management and Analysis", "C"),
    ("International Journal of Financial Studies", "C"),
    ("Journal of Finance and Investment Analysis", "C"),
    ("Journal of Applied Finance", "C"),
    ("Investment Management and Financial Innovations", "C"),
    ("African Journal of Economic and Management Studies", "C"),
    ("Journal of Financial Economic Policy", "C"),
    ("Australasian Accounting Business and Finance Journal", "C"),
    ("Asian Economic and Financial Review", "C"),
    ("Banks and Bank Systems", "C"),
    ("Journal of Emerging Market Finance", "C"),
    ("International Journal of Finance", "C"),
    ("Journal of Financial Reporting and Accounting", "C"),
    ("Studies in Economics and Finance", "C"),
    ("Review of Financial Economics", "C"),
    ("China Finance Review International", "C"),
    ("Cogent Economics and Finance", "C"),
    ("Journal of Risk Finance", "C"),
    ("Journal of Wealth Management", "C"),
]

finance_journals = pd.DataFrame(finance_journals_data, columns=["journal_name", "abdc_rank"])
print(f"Defined {len(finance_journals)} finance journals")
print(f"\nBy rank:")
print(finance_journals["abdc_rank"].value_counts().sort_index())

Defined 90 finance journals

By rank:
abdc_rank
A     30
A*    10
B     30
C     20
Name: count, dtype: int64


## 2. Match Journals to OpenAlex Source IDs

We search the OpenAlex Sources API for each journal name and use fuzzy matching to find the best match.

In [3]:
def fuzzy_match_score(a: str, b: str) -> float:
    """Compute similarity ratio between two strings."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


def search_openalex_source(journal_name: str) -> dict | None:
    """Search OpenAlex for a journal and return the best match."""
    try:
        results = Sources().search(journal_name).get()
        if not results:
            return None
        
        # Find the best fuzzy match
        best_match = None
        best_score = 0.0
        for source in results[:10]:  # Check top 10 results
            name = source.get("display_name", "")
            score = fuzzy_match_score(journal_name, name)
            if score > best_score:
                best_score = score
                best_match = source
        
        if best_match and best_score > 0.5:
            return {
                "source_id": best_match["id"],
                "openalex_name": best_match["display_name"],
                "issn_l": best_match.get("issn_l"),
                "works_count": best_match.get("works_count", 0),
                "match_score": best_score,
            }
        return None
    except Exception as e:
        print(f"  Error searching for '{journal_name}': {e}")
        return None


# Search for all journals
matched = []
unmatched = []

for _, row in tqdm(finance_journals.iterrows(), total=len(finance_journals), desc="Matching journals"):
    journal_name = row["journal_name"]
    abdc_rank = row["abdc_rank"]
    
    result = search_openalex_source(journal_name)
    
    if result:
        matched.append({
            "journal_name": journal_name,
            "abdc_rank": abdc_rank,
            "source_id": result["source_id"],
            "openalex_name": result["openalex_name"],
            "issn_l": result["issn_l"],
            "works_count": result["works_count"],
            "match_score": result["match_score"],
        })
    else:
        unmatched.append(journal_name)
    
    time.sleep(0.15)  # Respect rate limits

matched_df = pd.DataFrame(matched)
print(f"\nMatched: {len(matched)} / {len(finance_journals)}")
print(f"Unmatched: {len(unmatched)}")
if unmatched:
    print(f"\nUnmatched journals:")
    for j in unmatched:
        print(f"  - {j}")

Matching journals:   0%|          | 0/90 [00:00<?, ?it/s]


Matched: 90 / 90
Unmatched: 0


In [4]:
# Review matches — check for low-confidence matches
print("Matches with score < 0.8 (review these manually):\n")
low_confidence = matched_df[matched_df["match_score"] < 0.8]
if len(low_confidence) > 0:
    for _, row in low_confidence.iterrows():
        print(f"  {row['journal_name']}")
        print(f"    -> {row['openalex_name']} (score: {row['match_score']:.2f})")
        print()
else:
    print("  All matches have score >= 0.8")

print(f"\nTotal works across matched journals: {matched_df['works_count'].sum():,}")

Matches with score < 0.8 (review these manually):

  Review of Finance
    -> Brazilian Review of Finance (score: 0.77)


Total works across matched journals: 172,368


## 3. Save Journal List

In [5]:
# Save the matched journal list
output_path = "data/processed/journal_list.csv"
matched_df.to_csv(output_path, index=False)
print(f"Saved {len(matched_df)} journals to {output_path}")
print(f"\nJournals by ABDC rank:")
print(matched_df["abdc_rank"].value_counts().sort_index())
print(f"\nSample:")
matched_df.head(10)

Saved 90 journals to data/processed/journal_list.csv

Journals by ABDC rank:
abdc_rank
A     30
A*    10
B     30
C     20
Name: count, dtype: int64

Sample:


,journal_name,abdc_rank,source_id,openalex_name,issn_l,works_count,match_score
0,Journal of Finance,A*,https://openalex.org/S5353659,The Journal of Finance,0022-1082,16074,0.900000
1,Journal of Financial Economics,A*,https://openalex.org/S149240962,Journal of Financial Economics,0304-405X,4836,1.000000
2,Review of Financial Studies,A*,https://openalex.org/S170137484,Review of Financial Studies,0893-9454,3242,1.000000
3,Journal of Monetary Economics,A*,https://openalex.org/S6711363,Journal of Monetary Economics,0304-3932,3981,1.000000
4,Journal of Financial and Quantitative Analysis,A*,https://openalex.org/S193228710,Journal of Financial and Quantitative Analysis,0022-1090,4040,1.000000
5,Review of Finance,A*,https://openalex.org/S2765019445,Brazilian Review of Finance,1679-0731,442,0.772727
6,"Journal of Money, Credit and Banking",A*,https://openalex.org/S2058785,Journal of money credit and banking,0022-2879,4552,0.985915
7,Mathematical Finance,A*,https://openalex.org/S133335107,Mathematical Finance,0960-1627,1393,1.000000
8,Finance and Stochastics,A*,https://openalex.org/S172526255,Finance and Stochastics,0949-2984,819,1.000000
9,Journal of Financial Intermediation,A*,https://openalex.org/S5984737,Journal of Financial Intermediation,1042-9573,1026,1.000000
